# Cosmos DB vector search with LangChain

This notebook shows the higher-level LangChain pattern for retrieval-augmented generation. The main idea is the same as the manual notebook, but the code is packaged as reusable retrievers and chains.

Install the dependencies if needed:

```bash
pip install langchain langchain-openai langchain-community langchain-azure-ai
```

In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_azure_ai.vectorstores.azure_cosmos_db_no_sql import AzureCosmosDBNoSqlVectorSearch
from azure.cosmos import CosmosClient

In [ ]:
load_dotenv()

os.environ["AZURE_OPENAI_ENDPOINT"] = os.environ["AOAI_ENDPOINT"]
os.environ["AZURE_OPENAI_API_KEY"] = os.environ["AOAI_KEY"]
os.environ["AZURE_OPENAI_API_VERSION"] = os.environ["AOAI_API_VERSION"]

cosmos_client = CosmosClient.from_connection_string(
    os.environ["COSMOS_DB_CONNECTION_STRING"]
)

vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/contentVector",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 1536,
        }
    ]
}

indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [
        {"path": "/\"_etag\"/?"},
        {"path": "/contentVector/*"},
    ],
    "vectorIndexes": [{"path": "/contentVector", "type": "diskANN"}],
}

In [ ]:
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.environ["AOAI_ENDPOINT"],
    api_key=os.environ["AOAI_KEY"],
    api_version=os.environ["AOAI_API_VERSION"],
    azure_deployment=os.environ["EMBEDDINGS_DEPLOYMENT_NAME"],
)

vector_store = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client=cosmos_client,
    database_name="test_db_5",
    container_name="product_v",
    embedding=embeddings,
    vector_embedding_policy=vector_embedding_policy,
    indexing_policy=indexing_policy,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [ ]:
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AOAI_ENDPOINT"],
    api_key=os.environ["AOAI_KEY"],
    api_version=os.environ["AOAI_API_VERSION"],
    azure_deployment=os.environ.get("CHAT_DEPLOYMENT_NAME", "gpt-4o-mini"),
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful product assistant.
    Use only the context below to answer the question.
    If the answer is not in the context, say you do not know.

    Context:
    {context}

    Question: {question}
    """
)

chain = {
    "context": lambda x: "\n\n".join(doc.page_content for doc in retriever.invoke(x)),
    "question": lambda x: x,
} | prompt | llm | StrOutputParser()

In [ ]:
question = "Which product is best for hiking in rough terrain?"
answer = chain.invoke(question)
print(answer)

## Why this is different from the manual notebook

The manual notebook writes the retrieval and prompt-building logic explicitly.
LangChain packages those pieces into composable components: retriever, prompt template, model, and chain. This makes it easier to add tools, memory, and multi-step orchestration.